# **Similarity Score Across Resume PDFs**

## 1. Imports

In [9]:
from pathlib import Path
import math
import re
from collections import Counter

import numpy as np
import pandas as pd

try:
    from pypdf import PdfReader
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install the PDF parser first with: pip install pypdf") from exc

## 2. Load PDF Documents

In [10]:
DATA_DIR = Path("../data")
pdf_paths = sorted(DATA_DIR.glob("*.pdf"))

print(f"Found {len(pdf_paths)} PDF documents")
for path in pdf_paths:
    print("-", path.name)

Found 8 PDF documents
- business_analyst_1.pdf
- business_analyst_2.pdf
- business_analyst_3.pdf
- civil_1.pdf
- civil_2.pdf
- data_science.pdf
- db.pdf
- etl_dev.pdf


In [11]:
def extract_pdf_text(pdf_path):
    """Extract text from every page of one PDF file."""
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        pages.append(page.extract_text() or "")
    return "\n".join(pages)


extracted_documents = {path.stem: extract_pdf_text(path) for path in pdf_paths}

# these PDFs are image-only in this dataset, so pypdf cannot read their visible text.
# fallback below is a compact manual OCR transcription of the resume content.
manual_ocr_fallback = {
    "business_analyst_1": "Dedicated business analyst with strong understanding of finance operational proficiency business development project management trend analysis structure policies operations organization recommend solutions improve business processes. Business analyst Lexagon support process improvement customer service operations analyze define develop complex business process methods improve customer experience requirements expectations recommendations benchmark data test scripts accuracy loaded data financial analysis due diligence projects evaluate existing processes optimal solutions improvements. Skills business development management leadership problem solving attention to detail interpersonal skills finance accounting education NYU Stern School of Business finance business technology intern.",
    "business_analyst_2": "Dynamic entry level business analyst summary strong interest in IT consulting services define business and technical requirements based on business problems translating simplifying requirements optimizing execution outcomes agile waterfall methodologies. Business analyst intern XYZ company requirements continuous improvement project efficiency project execution development speed analytical skills communication skills adaptability business analysis requirements definition agile methodology waterfall methodology achievements business requirements project execution development speed computer science education.",
    "business_analyst_3": "Business analyst supporting analytical needs organizations project product management analytics expert. Work experience extracted data from Sabre Duetto Google Analytics into MS Excel pivot tables VLOOKUP macros analysis business impacting insights recommendations predictive analysis guests occupancy revenue e-commerce performance reports PowerPoint presentations KPI metrics business success cross functional teams fulfillment corporate goals productivity SQL queries joins subqueries database. Project manager intern JIRA Confluence data using IFTTT WeirdWeb Excel pivot tables graphs mockups wireframes use cases activity diagrams workflow developer Python HTML codes test plans BRDs FRDs. Technical skills Oracle MySQL Tableau MS Project Salesforce SharePoint scrum microsoft project IBM DB2 database administration fundamentals academic projects regression analysis data warehousing supply chain management operations data warehouse ETL cost quality data sources.",
    "civil_1": "Civil engineer graduate ABET accredited civil engineering program internship engineering theories principles specifications standards engineer in training. Skills AutoCAD Civil 3D MicroStation ArcGIS site layouts planning code zoning subdivision storm water ordinances. Education civil engineering design cost estimating surveying structural analysis dynamics geotechnical engineering construction methods traffic materials engineering environmental engineering water resource engineering fluid mechanics hydraulics concrete steel design. Professional experience civil engineering group roadway designs improvements traffic congestion bridges cost materials estimating report document tracking project documentation site visits building permits blueprint reading maps plans.",
    "civil_2": "Civil engineer seasoned experience designing constructing large scale infrastructure projects manage team engineers complete multimillion bridge project on time under budget AutoCAD drafting detailed design plans site data project documentation technical recommendations safety regulations. Employment civil engineer inspected monitored infrastructure roads bridges cost savings project completion design specifications budget constraints roadway improvements stakeholders city cycle construction survey maps calculations safety standards planning process cost change orders permits acquisition documentation quality control.",
    "data_science": "Data science engineering leader senior leadership drive organizational KPIs advanced data analytics artificial intelligence software products platforms. Principal director data science engineering strategic product thinking data driven decisioning SDLC planning cross functional coordination team development operations management. Expertise technology leadership product planning OKR definition MVP definition predictive analytics data science engineering machine learning distributed computing high performance computing optimization simulation algorithms data structures Java Python Spark MapReduce. Product portfolio data quality inspector recommendations simulator AI ChatOps bot personalized recommendations API rules engine feedback data collection impressions analysis operational monitoring SLA alerting forecasting visualization. Experience Target principal data science engineering roadmap KPIs Hitachi data systems SAN analytics reporting monitoring SLA management Carnegie Mellon electrical computer engineering computer science.",
    "db": "Database administrator DBA Oracle database installations normalization design creation maintenance database tuning RMAN backup recovery Automatic Storage Management Real Application Cluster Data Guard patching upgrade security management performance tuning UNIX Linux Windows. Skills RDBMS PostgreSQL Oracle Bash shell scripting SQL PL/PGSQL capacity monitoring standby failover administration data security backup recovery high availability migration upgrades. Work history monitoring database growth DWH databases table partitioning performance alerts uptime upgrades data migration index bloat maintenance Oracle Enterprise Manager PL SQL UNIX scripting CRONJOB AWR ADDM SQL trace TKPROF explain plan Data Pump export import backups PostgreSQL environments database definition structure capacity planning PgAdmin HAProxy AWS EC2 Terraform Ansible replication disaster recovery vacuum query response time database design modeling.",
    "etl_dev": "ETL developer shell scripting agile environment analytics automation Ab Initio GDE Co Operating System Talend Autosys Arrow Control M EME GIT Pac Quality Center data modeling star schema snowflake schema Erwin Oracle Microsoft SQL Server IBM DB2 UNIX Windows SQL PLSQL. Work history extracted data from databases delimited flat files Ab Initio components reformat scan rollup join sort partition normalize input output update table logs SQL graphs procedures applications Citrix servers data quality checks profiling configuration DML xfr files business logic Teradata BTEQ scripts data warehouse mainframe transformations EFS scheduling stored procedures UNIX scripts SQL optimization checkpoint process data validation backend testing cleansing AWS step functions lambda EC2 S3 IAM SNS quick reports file management oracle reporting computer science.",
}

documents = {}
fallback_used = []
for name, text in extracted_documents.items():
    if text.strip():
        documents[name] = text
    else:
        documents[name] = manual_ocr_fallback.get(name, "")
        fallback_used.append(name)

if fallback_used:
    print("Manual OCR fallback used for image-only PDFs:")
    for name in fallback_used:
        print("-", name)

if not any(text.strip() for text in documents.values()):
    raise ValueError("No document text was available. Add OCR text or use PDFs with selectable text.")

doc_summary = pd.DataFrame(
    {
        "document": list(documents.keys()),
        "characters": [len(text) for text in documents.values()],
        "raw_words": [len(text.split()) for text in documents.values()],
    }
)
doc_summary

Manual OCR fallback used for image-only PDFs:
- business_analyst_1
- business_analyst_2
- business_analyst_3
- civil_1
- civil_2
- data_science
- db
- etl_dev


,document,characters,raw_words
0,business_analyst_1,792,89
1,business_analyst_2,614,67
2,business_analyst_3,970,120
3,civil_1,774,87
4,civil_2,625,71
5,data_science,1042,117
6,db,932,115
7,etl_dev,852,119


## 3. Clean and Tokenize Text

In [12]:
STOP_WORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has",
    "he", "in", "is", "it", "its", "of", "on", "that", "the", "to", "was",
    "were", "will", "with", "or", "this", "these", "those", "their", "there",
    "into", "about", "using", "use", "used", "also", "can", "may", "such",
    "etc", "not", "than", "then", "they", "them", "his", "her", "you", "your",
    "we", "our", "i", "me", "my", "so", "if", "but", "while", "during",
    "within", "without", "over", "under", "between", "through", "per", "via",
}


def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-z]+", text)
    return [token for token in tokens if token not in STOP_WORDS and len(token) > 2]


tokenized_documents = {name: tokenize(text) for name, text in documents.items()}

pd.DataFrame(
    {
        "document": list(tokenized_documents.keys()),
        "clean_tokens": [len(tokens) for tokens in tokenized_documents.values()],
        "unique_terms": [len(set(tokens)) for tokens in tokenized_documents.values()],
    }
)

,document,clean_tokens,unique_terms
0,business_analyst_1,85,64
1,business_analyst_2,63,43
2,business_analyst_3,113,96
3,civil_1,85,66
4,civil_2,69,59
5,data_science,116,84
6,db,113,85
7,etl_dev,111,96


## 4. Build the Vector Space Model

In [13]:
document_names = list(tokenized_documents.keys())
vocabulary = sorted({term for tokens in tokenized_documents.values() for term in tokens})

term_counts = {name: Counter(tokens) for name, tokens in tokenized_documents.items()}
token_sets = {name: set(tokens) for name, tokens in tokenized_documents.items()}
document_frequency = {
    term: sum(1 for tokens in token_sets.values() if term in tokens)
    for term in vocabulary
}

num_documents = len(document_names)
idf = {
    term: math.log((1 + num_documents) / (1 + document_frequency[term])) + 1
    for term in vocabulary
}

tfidf_matrix = np.zeros((num_documents, len(vocabulary)))

for row, name in enumerate(document_names):
    counts = term_counts[name]
    total_terms = sum(counts.values())
    for col, term in enumerate(vocabulary):
        tf = counts[term] / total_terms if total_terms else 0
        tfidf_matrix[row, col] = tf * idf[term]

tfidf_df = pd.DataFrame(tfidf_matrix, index=document_names, columns=vocabulary)
print("TF-IDF matrix shape:", tfidf_df.shape)
tfidf_df.iloc[:, :10].round(4)

TF-IDF matrix shape: (8, 458)


,abet,academic,accounting,accredited,accuracy,achievements,acquisition,activity,adaptability,addm
business_analyst_1,0.0000,0.0000,0.0295,0.0000,0.0295,0.0000,0.0000,0.0000,0.0000,0.0000
business_analyst_2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0397,0.0000,0.0000,0.0397,0.0000
business_analyst_3,0.0000,0.0222,0.0000,0.0000,0.0000,0.0000,0.0000,0.0222,0.0000,0.0000
civil_1,0.0295,0.0000,0.0000,0.0295,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
civil_2,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0363,0.0000,0.0000,0.0000
data_science,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
db,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0222
etl_dev,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


## 5. Calculate Cosine Similarity

In [14]:
def cosine_similarity_matrix(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1
    normalized = matrix / norms
    return normalized @ normalized.T


similarity_matrix = cosine_similarity_matrix(tfidf_matrix)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=document_names,
    columns=document_names,
)

similarity_df.round(3)

,business_analyst_1,business_analyst_2,business_analyst_3,civil_1,civil_2,data_science,db,etl_dev
business_analyst_1,1.000,0.334,0.202,0.030,0.050,0.099,0.036,0.079
business_analyst_2,0.334,1.000,0.154,0.022,0.043,0.050,0.010,0.047
business_analyst_3,0.202,0.154,1.000,0.036,0.102,0.193,0.163,0.171
civil_1,0.030,0.022,0.036,1.000,0.261,0.201,0.019,0.000
civil_2,0.050,0.043,0.102,0.261,1.000,0.053,0.041,0.037
data_science,0.099,0.050,0.193,0.201,0.053,1.000,0.113,0.151
db,0.036,0.010,0.163,0.019,0.041,0.113,1.000,0.185
etl_dev,0.079,0.047,0.171,0.000,0.037,0.151,0.185,1.000


## 6. Rank Document Pairs by Similarity

In [15]:
pair_scores = []

for i, doc_a in enumerate(document_names):
    for j in range(i + 1, len(document_names)):
        doc_b = document_names[j]
        pair_scores.append(
            {
                "document_1": doc_a,
                "document_2": doc_b,
                "cosine_similarity": similarity_matrix[i, j],
            }
        )

pair_scores_df = pd.DataFrame(pair_scores).sort_values(
    "cosine_similarity", ascending=False
)

pair_scores_df.round({"cosine_similarity": 3})

,document_1,document_2,cosine_similarity
0,business_analyst_1,business_analyst_2,0.334
18,civil_1,civil_2,0.261
1,business_analyst_1,business_analyst_3,0.202
19,civil_1,data_science,0.201
15,business_analyst_3,data_science,0.193
27,db,etl_dev,0.185
17,business_analyst_3,etl_dev,0.171
16,business_analyst_3,db,0.163
7,business_analyst_2,business_analyst_3,0.154
26,data_science,etl_dev,0.151


## 7. Results Summary

In [16]:
most_similar = pair_scores_df.iloc[0]
least_similar = pair_scores_df.iloc[-1]

print("Most similar pair:")
print(f"{most_similar.document_1} <-> {most_similar.document_2}: {most_similar.cosine_similarity:.3f}")

print("\nLeast similar pair:")
print(f"{least_similar.document_1} <-> {least_similar.document_2}: {least_similar.cosine_similarity:.3f}")

print("\nTop 5 most similar pairs:")
display(pair_scores_df.head(5).round({"cosine_similarity": 3}))

Most similar pair:
business_analyst_1 <-> business_analyst_2: 0.334

Least similar pair:
civil_1 <-> etl_dev: 0.000

Top 5 most similar pairs:


,document_1,document_2,cosine_similarity
0,business_analyst_1,business_analyst_2,0.334
18,civil_1,civil_2,0.261
1,business_analyst_1,business_analyst_3,0.202
19,civil_1,data_science,0.201
15,business_analyst_3,data_science,0.193
